In [2]:
import glob
import os

import pandas as pd

series = []

for path in sorted(glob.glob("*.csv")):
    name = os.path.basename(path)
    if name == "predictors_monthly.csv":
        continue
    df = pd.read_csv(path)
    df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])
    df = df.set_index(df.columns[0]).sort_index()
    df.index.name = "date"
    df = df.dropna(how="all")

    if (df.index.day == 1).mean() > 0.9:
        # start-of-month stamp: lag one month (Mar 1 -> Feb 28)
        df.index = df.index - pd.Timedelta(days=1)
    elif df.index.to_series().diff().dt.days.median() < 5:
        # daily data: last observation of each month; drop the incomplete final month
        last = df.index.max()
        df = df.groupby(df.index.to_period("M")).last()
        if last != last + pd.offsets.MonthEnd(0):
            df = df.iloc[:-1]
        df.index = df.index.to_timestamp(how="end").normalize()
    # month-end / quarter-end stamps are already fine
    df.index = df.index + pd.offsets.MonthEnd(0)
    series.append(df)

predictors = pd.concat(series, axis=1).sort_index()
predictors.index.name = "date"
predictors.to_csv("predictors_monthly.csv")

print(predictors.shape)
print(predictors.apply(lambda s: s.first_valid_index()))
predictors.tail()

(1294, 16)
AAA                          1918-12-31
BAA                          1918-12-31
CPIAUCSL                     1946-12-31
DCOILWTICO                   1986-01-31
GDPC1                        1946-12-31
GS10                         1953-03-31
INDPRO                       1918-12-31
vwretd                       1926-01-31
vwretx                       1926-01-31
TB3MS                        1933-12-31
UNRATE                       1947-12-31
VIXCLS                       1990-01-31
c                            1952-03-31
w                            1952-03-31
y                            1952-03-31
cay=c-0.218 a-0.801y+0.441   1952-03-31
dtype: datetime64[ns]


,AAA,BAA,CPIAUCSL,DCOILWTICO,GDPC1,GS10,INDPRO,vwretd,vwretx,TB3MS,UNRATE,VIXCLS,c,w,y,cay=c-0.218 a-0.801y+0.441
date,,,,,,,,,,,,,,,,
2026-05-31,5.52,6.00,332.568,91.16,NaN,4.47,102.9610,NaN,NaN,3.66,4.2,15.32,NaN,NaN,NaN,NaN
2026-06-30,5.76,6.19,332.813,70.56,NaN,4.60,103.2006,NaN,NaN,3.73,4.1,16.45,NaN,NaN,NaN,NaN
2026-07-31,5.88,6.32,334.131,86.16,NaN,4.68,103.2664,NaN,NaN,3.72,4.1,15.99,NaN,NaN,NaN,NaN
2026-08-31,6.03,6.47,336.298,87.03,NaN,4.99,103.9506,NaN,NaN,3.94,4.2,14.92,NaN,NaN,NaN,NaN
2026-09-30,NaN,NaN,NaN,97.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.34,NaN,NaN,NaN,NaN


In [ ]:
"""Block 2 in the Data: Risk and Risk Aversion (standalone).

Reproduces the slide's risk block from this folder's data:
  VIXCLS.csv             (FRED VIX, daily -> month-end)
  predictors_monthly.csv (FRED + Shiller: TB3MS, vwretd, cay)

Prediction: higher volatility or risk aversion -> higher future returns.
Table: correlation of each variable today with future EXCESS stock
returns at 1 quarter / 1 year / 2 years / 5 years; Newey-West t-statistic
in brackets; |t| > 2 is statistically significant.

Self-contained: the helpers below are defined in this cell, so it can be
read and run on its own.
"""

from pathlib import Path

import numpy as np
import pandas as pd

HORIZONS = (3, 12, 24, 60)  # months: 1Q, 1Y, 2Y, 5Y
HLAB = {3: "1Q", 12: "1Y", 24: "2Y", 60: "5Y"}
DATA_DIRS = (".", "Predictor Data", "Market Return Predictors/Predictor Data", "data")

# Slide values: (correlation, t-statistic) per horizon.
SLIDE = {
    "VIX": {3: (0.12, 0.96), 12: (0.10, 0.63), 24: (0.13, 0.75), 60: (-0.16, -0.47)},
    "Past volatility (last quarter)": {3: (0.05, 0.55), 12: (0.09, 0.65), 24: (0.13, 0.78), 60: (0.06, 0.17)},
    "Consumption-wealth ratio": {3: (0.16, 2.92), 12: (0.30, 2.62), 24: (0.42, 2.23), 60: (0.53, 1.34)},
}


def find_csv(name, dirs=DATA_DIRS):
    path = next((Path(d, name) for d in dirs if Path(d, name).exists()), None)
    if path is None:
        raise FileNotFoundError(f"{name} not found in {dirs}")
    return path


def future_returns(r, horizons=HORIZONS):
    """Overlapping compounded returns: R(t,h) = prod(1 + r_{t+1..t+h}) - 1.

    Log-return rolling sum is backward-looking ([t-h+1, t]); shifting by -h
    months gives the forward window [t+1, t+h].
    """
    out = {}
    logr = np.log1p(r)
    for h in horizons:
        out[f"R{h}"] = np.expm1(logr.rolling(h).sum().shift(-h))
    return pd.DataFrame(out, index=r.index)


def corr_tstat_nw(x, y, nw_lag):
    """Correlation of x and y with a Newey-West (HAC) t-statistic.

    Standardizes both series; the OLS slope of y on x is the correlation.
    Its standard error uses a Bartlett-kernel HAC covariance with `nw_lag`
    lags and the n/(n-k) small-sample correction. Returns (rho, tstat, nobs).
    """
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    n = len(d)
    if n < 30:
        return np.nan, np.nan, n
    xs = (d["x"] - d["x"].mean()) / d["x"].std()
    ys = (d["y"] - d["y"].mean()) / d["y"].std()
    z = np.column_stack([np.ones(n), xs.to_numpy()])
    yv = ys.to_numpy()
    b = np.linalg.lstsq(z, yv, rcond=None)[0]
    zte = z * (yv - z @ b)[:, None]
    s = zte.T @ zte
    for lag in range(1, nw_lag + 1):
        w = 1.0 - lag / (nw_lag + 1)
        g = zte[lag:].T @ zte[:-lag]
        s += w * (g + g.T)
    bread = np.linalg.inv(z.T @ z)
    cov = bread @ s @ bread * (n / (n - z.shape[1]))
    return float(d["x"].corr(d["y"])), float(b[1] / np.sqrt(cov[1, 1])), n


def winsorize(s, lo=0.01, hi=0.99):
    return s.clip(*s.quantile([lo, hi]))


def load_monthly_vix(name="VIXCLS.csv"):
    """FRED VIXCLS.csv daily closes -> month-end, same rule as the build:
    last quote of each month, dropping an incomplete final month.
    """
    vix = pd.read_csv(find_csv(name), parse_dates=["observation_date"])
    vix = vix.set_index("observation_date")["VIXCLS"].dropna().sort_index()
    last = vix.index.max()
    vix = vix.groupby(vix.index.to_period("M")).last()
    if last != last + pd.offsets.MonthEnd(0):
        vix = vix.iloc[:-1]
    vix.index = vix.index.to_timestamp(how="end").normalize()
    vix.name = "vix"
    return vix


def main():
    pred = pd.read_csv(find_csv("predictors_monthly.csv"), index_col=0,
                       parse_dates=True)
    pred.index = pred.index.to_period("M").to_timestamp("M")

    # VIX imported straight from FRED's VIXCLS.csv (month-end observations).
    vix = load_monthly_vix()
    ret = pred["vwretd"]  # CRSP value-weighted total return, monthly
    cay = pred[next(c for c in pred.columns if c.startswith("cay"))]

    # Slide predictor: past volatility (last quarter) = std of monthly total
    # returns over months t-2..t (correlation is scale-free, no annualizing).
    past_vol = ret.rolling(3).std()

    # Future EXCESS returns: total return minus 3-month T-bill rate.
    rf_m = pred["TB3MS"] / 1200
    fut_x = future_returns(ret - rf_m)

    block = {"VIX": winsorize(vix.dropna()),
             "Past volatility (last quarter)": winsorize(past_vol.dropna()),
             "Consumption-wealth ratio": winsorize(cay.dropna())}

    print(f"{'Predictor':32s}" + "".join(f"{HLAB[h]:>18s}" for h in HORIZONS))
    for name, x in block.items():
        cells = []
        for h in HORIZONS:
            rho, t, n = corr_tstat_nw(x, fut_x[f"R{h}"], h - 1)
            flag = "*" if abs(t) > 2 else " "
            cells.append(f"{rho:+.2f} ({t:+.2f}){flag} [n={n}]")
        print(f"{name:32s}" + "".join(f"{c:>18s}" for c in cells))
        scells = [f"{SLIDE[name][h][0]:+.2f} ({SLIDE[name][h][1]:+.2f})"
                  for h in HORIZONS]
        print(f"{'  slide':32s}" + "".join(f"{c:>18s}" for c in scells))
    print("\n* = |t| > 2, statistically significant.")


if __name__ == "__main__":
    main()
